In [4]:
%pwd


'c:\\Users\\shran\\OneDrive\\Desktop\\medical-chatbot\\medical-chatbot\\research'

In [5]:
import os 
os.chdir("../")
%pwd


'c:\\Users\\shran\\OneDrive\\Desktop\\medical-chatbot\\medical-chatbot'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\shran\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
#extract text from pdf file (path as input)
def load_pdf_files(data):
    loader = DirectoryLoader(
        data, 
        glob="*.pdf", 
        show_progress=True, 
        loader_cls=PyPDFLoader
        )
    documents = loader.load()
    return documents

In [8]:
extracted_data = load_pdf_files("data")

100%|██████████| 1/1 [00:27<00:00, 27.34s/it]


In [9]:
len(extracted_data)
#extracted_data

637

In [10]:
#filter operation
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    given a list of documnent objects, return a new list of document objects with 
    only 'SOURCE' in metadata and the original page content 
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src= doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content, 
                metadata={"source": src}
            )
        )
    return minimal_docs


In [11]:
minimal_docs = filter_to_minimal_docs(extracted_data)
minimal_docs[10]

Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='Rhonda Cloos, R.N.\nMedical Writer\nAustin, TX\nGloria Cooksey, C.N.E\nMedical Writer\nSacramento, CA\nAmy Cooper, M.A., M.S.I.\nMedical Writer\nVermillion, SD\nDavid A. Cramer, M.D.\nMedical Writer\nChicago, IL\nEsther Csapo Rastega, R.N., B.S.N.\nMedical Writer\nHolbrook, MA\nArnold Cua, M.D.\nPhysician\nBrooklyn, NY\nTish Davidson, A.M.\nMedical Writer\nFremont, California\nDominic De Bellis, Ph.D.\nMedical Writer/Editor\nMahopac, NY\nLori De Milto\nMedical Writer\nSicklerville, NJ\nRobert S. Dinsmoor\nMedical Writer\nSouth Hamilton, MA\nStephanie Dionne, B.S.\nMedical Writer\nAnn Arbor, MI\nMartin W. Dodge, Ph.D.\nTechnical Writer/Editor\nCentinela Hospital and Medical\nCenter\nInglewood, CA\nDavid Doermann\nMedical Writer\nSalt Lake City, UT\nStefanie B. N. Dugan, M.S.\nGenetic Counselor\nMilwaukee, WI\nDoug Dupler, M.A.\nScience Writer\nBoulder, CO\nJulie A. Gelderloos\nBiomedical Writer\nPlaya del Rey, CA\nGar

In [12]:
#chunking operation
#split the documents into smaller chunks (for better processing)
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, #500 token one chunk
        chunk_overlap=20, #for understanding the context, we need to have some overlap between chunks
        )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [13]:
texts_chunk = text_split(minimal_docs)
print(len(texts_chunk))

5859


In [14]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """download and return the embedding model"""
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings = download_embeddings()

C:\Users\shran\AppData\Local\Temp\ipykernel_15796\3975655537.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


In [15]:
vector=embeddings.embed_query("Hello world ") 
len(vector)

384

In [16]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [30]:
from pinecone import Pinecone
pinecone_api_key= PINECONE_API_KEY

pc=Pinecone(api_key=pinecone_api_key)
pc

In [31]:
from pinecone import ServerlessSpec
index_name = "medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, #dimension of the embedding vector
        metric="cosine", #similarity metric
        spec=ServerlessSpec(cloud="aws", region="us-east-1") #serverless configuration
    )

index=pc.Index(index_name)

In [32]:
from langchain_pinecone import PineconeVectorStore
docsearch= PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embeddings, 
    index_name=index_name
    )

In [33]:
#load existing index
from langchain_pinecone import PineconeVectorStore
#embed each chunk and store in the pineconeindex
docsearch= PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name
    )

In [34]:
#retriever
#retrieve relevant documents based on a query
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [35]:
retrieved_docs = retriever.invoke("What are the symptoms of diabetes?")
retrieved_docs

[Document(id='8bc94b16-ba4a-4e34-ad32-91d82943cf7c', metadata={'source': 'data\\Medical_book.pdf'}, page_content='• Type I diabetes mellitus. Characterized by fatigue and\nan abnormally high level of glucose in the blood\n(hyperglycemia).\n• Amyotrophic lateral schlerosis. First signs are stum-\nbling and difficulty climbing stairs. Later, muscle\ncramps and twitching may be observed as well as\nweakness in the hands making fastening buttons or\nturning a key difficult. Speech may become slowed or\nslurred. There may also be difficluty swallowing. As\nrespiratory muscles atrophy, there is increased danger'),
 Document(id='cc9beda8-461b-45af-862a-223e43beca38', metadata={'source': 'data\\Medical_book.pdf'}, page_content='begin to fall. A person with diabetes mellitus either does\nnot make enough insulin, or makes insulin that does not\nwork properly. The result is blood sugar that remains\nhigh, a condition called hyperglycemia.\nDiabetes must be diagnosed as early as possible. If\nleft

In [37]:
#importing the llm- to frame the retrieved documents into a coherent answer

from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4o")

In [38]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


In [40]:
#prompt template
system_prompt=(
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [41]:
#create a chain that combines the retrieved documents and the prompt template
question_answer_chain = create_stuff_documents_chain(chat_model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [42]:
#ask a question to the RAG chain
response = rag_chain.invoke({"input": "What are the symptoms of diabetes?"})
print(response["answer"])

The symptoms of diabetes include fatigue and abnormally high blood glucose levels (hyperglycemia). If left untreated, diabetes can cause damage to the eyes, kidneys, nerves, heart, blood vessels, and other organs.
